In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np 
import chromadb

/tmp/ipykernel_29224/1728783111.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
chromadb_client = chromadb.PersistentClient(path="/home/wayne-ramsey/Desktop/CLASSWORK/ollama_basics/data/chromadb")

#### intro to evaluations

In [3]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])

In [4]:
# Dot Product Similarity
dot_sim = np.dot(a, b)

In [5]:
# Cosine Similarity
cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
# Euclidean Distance (Lower is better)
euclidean_dist = np.linalg.norm(a - b)   

In [7]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

 
 DOT PRODUCT SIM_SCORE: 14  
 
 COSINE SIM_SCORE: 1.0  
 
 EUCLIDEAN DISTANCE: 0.0  
 



In [8]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [9]:
text_one  = "my dog is called chucho"
text_two = "my cat is called potpot"

In [10]:
def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings

In [11]:
text_one_embeding = createEmbedding(text=text_one)
text_two_embeding = createEmbedding(text=text_two)

In [12]:
np.array(text_one_embeding[0][0:10])

array([ 0.0183344 ,  0.07860761, -0.02505185, -0.00841216, -0.02814166,
        0.02554682,  0.02493315, -0.01011948,  0.02449936,  0.00368196])

In [13]:
print(text_one_embeding)
print(text_two_embeding)

[[0.018334402, 0.07860761, -0.025051849, -0.008412158, -0.028141662, 0.025546823, 0.024933146, -0.010119482, 0.024499357, 0.0036819563, -0.0005142216, 0.01135153, 0.048144888, -0.059075788, 0.019159153, -0.0044522705, -0.024141388, -0.07620838, 0.061696254, 0.03831744, 0.05134889, 0.03105408, 0.023972638, -0.027483905, 0.018543823, 0.11562764, 0.060294725, 0.038280547, 0.06528984, -0.04946809, -0.060224332, -0.007700296, -0.097674444, 0.0021032253, -0.046798196, -0.018698977, 0.0022781193, -0.0064445203, 0.0674335, -0.034116548, 0.044858575, 0.019791823, -0.023729468, 0.04978651, 0.037794888, -0.037155148, -0.002091275, -0.006812096, 0.005787611, -0.057599746, 0.01883346, -0.03292746, -0.011379694, -0.004394732, -0.0023798577, -0.11846551, 0.027489308, 0.040744286, 0.05769931, 0.00558665, 0.03196524, 0.04334708, -0.022287397, -0.033921015, -0.022745617, -0.012997379, -0.017879095, 0.026398716, 0.052118886, -0.01595977, 0.0015965494, -0.0005400083, -0.0542784, 0.058608443, -0.072285056,

In [14]:
similarityChecks(text_one_embeding= text_one_embeding[0], text_two_embeding=text_two_embeding[0])
#if euclidean is closer to 0, the two texts are more similar. If the cosine similarity is closer to 1, the two texts are more similar. If the dot product is higher, the two texts are more similar.

  DOT PRODUCT SIM_SCORE: 0.376907781024741  
  COSINE SIM_SCORE: 0.3769078727053372  
  EUCLIDEAN DISTANCE: 1.1163260954857401  
 


In [15]:
collection = chromadb_client.create_collection(name="research_collection")

In [16]:
collection.add(
    ids=["1", "2"],
    embeddings=[text_one_embeding[0], text_two_embeding[0]],
)

In [17]:
results = collection.query(
    query_embeddings=[text_one_embeding[0], text_two_embeding[0]],
    n_results=2
)
print(results)

{'ids': [['1', '2'], ['2', '1']], 'embeddings': None, 'documents': [[None, None], [None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None], [None, None]], 'distances': [[0.0, 1.2461838722229004], [0.0, 1.2461838722229004]]}


In [18]:
loader = PyPDFLoader("data/3rd term evemon.pdf")
pages = loader.load()

In [19]:
pages

[Document(metadata={'producer': 'LibreOffice 26.2.5.2 (X86_64)', 'creator': 'Writer', 'creationdate': '2026-10-02T06:58:01+03:00', 'source': 'data/3rd term evemon.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Evemon Education Centre\nP.O BOX 15044 – 00400\nTel: 0710 987 197, 0142 290 377\nMOTTO : A GREAT STEP FOR A BETTER TOMORROW\nDear Parent/Guardian,\nGreetings in the name of Jesus. We give thanks to the LORD Almighty with grateful hearts for how far He has \nbrought us, and we trust Him to guide us through to the end of this term.\nEverything is progressing smoothly, and we would like to share the following updates:\n1. Mid-Term Examinations\nPre-school and Grades 1, 2, 3, and 6 have completed their mid-term examinations. Please go through the results \ntogether with your child.\n2. KNEC Assessment\nThe theory assessments for Grades 3, 4, and 5 begins today.\n3. Graduation & Closing Ceremony\nThe PP2 graduation ceremony will take place on the 22nd , which wil

In [20]:
def clean_text(text):
    cleaned_text = re.sub(r'\s+', ' ', text)  # Replace multiple whitespace with a single space
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s.@,!?;:()\-\'"]', '', cleaned_text)  # Remove special characters except punctuation
    return cleaned_text.strip()  # Remove leading and trailing whitespace

In [21]:
for page in pages:
    page.page_content = clean_text(page.page_content)

In [22]:
pages[0].page_content

'Evemon Education Centre P.O BOX 15044  00400 Tel: 0710 987 197, 0142 290 377 MOTTO : A GREAT STEP FOR A BETTER TOMORROW Dear ParentGuardian, Greetings in the name of Jesus. We give thanks to the LORD Almighty with grateful hearts for how far He has brought us, and we trust Him to guide us through to the end of this term. Everything is progressing smoothly, and we would like to share the following updates: 1. Mid-Term Examinations Pre-school and Grades 1, 2, 3, and 6 have completed their mid-term examinations. Please go through the results together with your child. 2. KNEC Assessment The theory assessments for Grades 3, 4, and 5 begins today. 3. Graduation  Closing Ceremony The PP2 graduation ceremony will take place on the 22nd , which will also be our closing date for Term 3.  PP2 parents who have not yet paid the KES 2,500 graduation fee are kindly requested to do so by October 10th.  We also request all other learners to contribute KES 100 to support the event.  Venue: School Groun

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=50,)

In [24]:
# text = pages[0].page_content

In [25]:
chunks = splitter.split_documents(pages)

In [26]:
chunks

[Document(metadata={'producer': 'LibreOffice 26.2.5.2 (X86_64)', 'creator': 'Writer', 'creationdate': '2026-10-02T06:58:01+03:00', 'source': 'data/3rd term evemon.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Evemon Education Centre P.O BOX 15044  00400 Tel: 0710 987 197, 0142 290 377 MOTTO : A GREAT STEP FOR A BETTER TOMORROW Dear ParentGuardian, Greetings'),
 Document(metadata={'producer': 'LibreOffice 26.2.5.2 (X86_64)', 'creator': 'Writer', 'creationdate': '2026-10-02T06:58:01+03:00', 'source': 'data/3rd term evemon.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='A BETTER TOMORROW Dear ParentGuardian, Greetings in the name of Jesus. We give thanks to the LORD Almighty with grateful hearts for how far He has'),
 Document(metadata={'producer': 'LibreOffice 26.2.5.2 (X86_64)', 'creator': 'Writer', 'creationdate': '2026-10-02T06:58:01+03:00', 'source': 'data/3rd term evemon.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='

In [28]:
doc_embeddings = []

for chunk in chunks:
    embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(embedding)

In [29]:
collection_research = chromadb_client.create_collection(name="evemon_documentation")

In [35]:
# collection_research = chromadb_client.create_collection(name="evemon_documentation")
for index in range(len(chunks)):
    collection_research.add(
        ids=[str(index)],
        documents=[chunks[index].page_content],
        embeddings=[doc_embeddings[index][0]]
    )

In [36]:
response = collection_research.query(
    query_texts="When is the graduation date?",
    query_embeddings= createEmbedding(text="When is the graduation date?"),
    n_results=3
)

In [38]:
response

{'ids': [['6', '7', '8']],
 'embeddings': None,
 'documents': [['assessments for Grades 3, 4, and 5 begins today. 3. Graduation  Closing Ceremony The PP2 graduation ceremony will take place on the 22nd , which will',
   'ceremony will take place on the 22nd , which will also be our closing date for Term 3.  PP2 parents who have not yet paid the KES 2,500 graduation',
   'who have not yet paid the KES 2,500 graduation fee are kindly requested to do so by October 10th.  We also request all other learners to contribute']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'page_label': '1',
    'producer': 'LibreOffice 26.2.5.2 (X86_64)',
    'total_pages': 1,
    'creator': 'Writer',
    'page': 0,
    'creationdate': '2026-10-02T06:58:01+03:00',
    'source': 'data/3rd term evemon.pdf'},
   {'producer': 'LibreOffice 26.2.5.2 (X86_64)',
    'total_pages': 1,
    'source': 'data/3rd term evemon.pdf',
    'page': 0,
    'creationdate': '20

In [39]:
for index in range(len(chunks)):
    collection_research.add(
        ids=[str(index)],
        documents = [chunks[index].page_content],                      
        embeddings=[doc_embeddings[index][0]],
        metadatas=[chunks[index].metadata],
    )

In [40]:
response['documents']

[['assessments for Grades 3, 4, and 5 begins today. 3. Graduation  Closing Ceremony The PP2 graduation ceremony will take place on the 22nd , which will',
  'ceremony will take place on the 22nd , which will also be our closing date for Term 3.  PP2 parents who have not yet paid the KES 2,500 graduation',
  'who have not yet paid the KES 2,500 graduation fee are kindly requested to do so by October 10th.  We also request all other learners to contribute']]